# Simple RAG with Filtering + Query Transformations

## 적용 기법
- **Metadata Filtering** : cate_depth1 기반 장르 필터
- **Query Transformations** : Sub-query Decomposition → 서브쿼리별 검색결합

## 0. 설치 및 초기화

In [74]:
!git clone https://github.com/jjeong3150/AIFFEL_final_pjt_book-recommendation-agent

fatal: destination path 'AIFFEL_final_pjt_book-recommendation-agent' already exists and is not an empty directory.


In [75]:
!pip install -q qdrant-client
!pip install -q langchain
!pip install -q langchain-openai
!pip install -q langchain-naver
!pip install -q FlagEmbedding

In [76]:
%run /content/AIFFEL_final_pjt_book-recommendation-agent/src/state/state_v1.ipynb
%run /content/AIFFEL_final_pjt_book-recommendation-agent/src/db/qdrant.py
%run /content/AIFFEL_final_pjt_book-recommendation-agent/src/embedding/embedder.py

In [77]:
import os
from google.colab import userdata

os.environ["OPENAI_API_KEY"]    = userdata.get('OPENAI_API_KEY')
os.environ["CLOVASTUDIO_API_KEY"] = userdata.get('CLOVASTUDIO_API_KEY')
os.environ["QDRANT_API_KEY"]    = userdata.get('QDRANT_API_KEY')
os.environ["QDRANT_URL"]        = userdata.get('QDRANT_URL')

In [78]:
from langchain_openai import ChatOpenAI
from langchain_naver import ChatClovaX

embedder = LocalEmbedder("BAAI/bge-m3")
db       = QdrantDB(vector_size=1024)
llm      = ChatOpenAI(model="gpt-4o-mini")
# llm    = ChatClovaX(model="HCX-005", temperature=0)

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

## 1. 장르 추출 노드 (simple_rag_with_filtering과 동일)

In [79]:
from langgraph.graph import StateGraph, START, END
from langchain_core.messages import HumanMessage, AIMessage
from langchain_core.prompts import ChatPromptTemplate
from qdrant_client.models import Filter, FieldCondition, MatchAny
import json

CATEGORY_LIST = ["소설", "대학교재/전문서적", "어린이", "수험서/자격증", "시/에세이", "종교", "유아", "만화",
                 "사회/정치", "경제/경영", "인문", "예술/대중문화", "국어/외국어", "고등학교 참고서",
                 "자기계발", "초등학교 참고서", "건강/취미", "컴퓨터/IT", "역사", "자연/과학",
                 "청소년", "중학교 참고서", "가정/요리", "여행", "잡지", "전집", "외국도서"]

genre_prompt = ChatPromptTemplate.from_template("""
사용자 프로파일을 보고 아래 카테고리 목록에서 적합한 것을 최대 3개 선택하세요.
목록에 없는 값은 절대 반환하지 마세요.

카테고리 목록: {category_list}
사용자 프로파일: {summary}

JSON으로만 반환: {{"categories": ["소설"]}}
""")

def extract_genre_node(state: CRSState) -> dict:
    summary = state.get("summary", "")
    chain = genre_prompt | llm
    response = chain.invoke({
        "category_list": CATEGORY_LIST,
        "summary": summary
    })
    try:
        categories = json.loads(response.content)["categories"]
    except (json.JSONDecodeError, KeyError):
        categories = []
    print(f"추출된 장르: {categories}")
    return {"genre_filter": categories}

## 2. Step-back Prompting
**입력**: 원본 사용자 프로파일  
**출력**: 추상화된 독서 목적  
**목적**: 구체적인 프로파일에서 한 단계 물러나 독자 타겟·독서 목적을 추상화 → 의도 파악 문제 해결

In [80]:
step_back_prompt = ChatPromptTemplate.from_template("""
당신은 도서 추천 시스템의 AI 어시스턴트입니다.
아래 사용자 프로파일에서 한 단계 물러나,
이 사용자가 독서를 통해 진정으로 원하는 것이 무엇인지
독자 타겟과 독서 목적 중심으로 추상화하세요.

[주의]
- 2문장 이내로 작성하세요.

사용자 프로파일: {summary}

출력:
""")

def step_back_query(summary: str, llm) -> str:
    """
    사용자 프로파일 → 추상화된 독서 목적

    Args:
        summary : 원본 사용자 프로파일
        llm     : LLM 인스턴스

    Returns:
        str: 추상화된 독서 목적
    """
    chain = step_back_prompt | llm
    return chain.invoke({"summary": summary}).content.strip()

## 3. Query Rewriting
**입력**: step_back 결과 (추상화된 독서 목적)  
**출력**: 검색에 최적화된 구체적인 쿼리  
**목적**: 추상화된 의도를 도서 검색에 더 적합한 표현으로 구체화

In [81]:
rewrite_prompt = ChatPromptTemplate.from_template("""
당신은 도서 추천 시스템의 AI 어시스턴트입니다.
아래 독서 목적을 도서 검색에 더 적합하고 구체적인 검색어로 재작성하세요.
장르, 독자 수준, 분위기 등 검색 정확도를 높일 수 있는 표현을 포함하세요.

독서 목적: {step_back}

재작성된 검색 쿼리 (한 문장으로):
""")

def rewrite_query(step_back: str, llm) -> str:
    """
    추상화된 독서 목적 → 검색에 최적화된 구체적 쿼리

    Args:
        step_back : step_back_query()의 결과
        llm       : LLM 인스턴스

    Returns:
        str: 재작성된 검색 쿼리
    """
    chain = rewrite_prompt | llm
    return chain.invoke({"step_back": step_back}).content.strip()

## 4. Sub-query Decomposition
**입력**: rewritten 결과 (구체화된 쿼리)  
**출력**: 조건 단위로 분해된 서브쿼리 리스트  
**목적**: 다중 조건을 분해하여 각각 검색 → RRF 결합으로 조건 만족률 향상

In [82]:
# decompose_prompt = ChatPromptTemplate.from_template("""
# 당신은 도서 추천 시스템의 AI 어시스턴트입니다.
# 아래 검색 쿼리를 도서 검색에 활용할 2~4개의 핵심 서브쿼리로 분해하세요.
# 각 서브쿼리는 장르, 분위기, 난이도, 독자 타겟 등 서로 다른 조건을 담아야 합니다.

# 검색 쿼리: {rewritten}

# 출력 형식 (번호와 텍스트만, 다른 텍스트 없이):
# 1. [서브쿼리 1]
# 2. [서브쿼리 2]
# 3. [서브쿼리 3]
# """)

# decompose_prompt = ChatPromptTemplate.from_template("""
# 당신은 도서 추천 시스템의 AI 어시스턴트입니다.
# 아래 검색 쿼리를 2~4개의 서브쿼리로 분해하세요.
# 서브쿼리는 원래 쿼리의 조건을 충실히 반영하되,
# 사용자가 미처 생각하지 못했을 새로운 관점이나 인사이트를 한 개 포함하세요.
# 각 서브쿼리는 독립적인 문장으로 작성하세요.

# [주의]
# - 리뷰, 평점, 사용자 의견 등 도서 메타데이터 외의 정보를 요청하지 마세요.
# - "추천 도서", "추천해주세요", "포함해주세요" 같은 표현으로 끝내지 마세요.

# 검색 쿼리: {rewritten}

# 출력 형식 (번호와 텍스트만, 다른 텍스트 없이):
# 1. [서브쿼리 1]
# 2. [서브쿼리 2]
# 3. [서브쿼리 3]
# """)


decompose_prompt = ChatPromptTemplate.from_template("""
당신은 도서 추천 시스템의 검색 쿼리 전문가입니다.
사용자의 독서 취향과 상황을 깊이 이해하여, 벡터 임베딩 검색에 최적화된 서브쿼리를 생성합니다.

아래 검색 쿼리를 2~4개의 서브쿼리로 분해하세요.
각 서브쿼리는 독립적인 문장으로 작성하되, 전체적으로 자연스럽게 맥락이 이어지도록 하세요.
원래 쿼리의 조건을 충실히 반영하면서, 사용자가 미처 생각하지 못했을 관련 관점을 한 개 포함하세요.


[주의]
- "이 중에서", "그 중에서" 같은 참조 표현은 사용하지 마세요.
- 리뷰, 평점, 사용자 의견 등 도서 메타데이터 외의 정보를 요청하지 마세요.
- "추천 도서", "추천해주세요", "포함해주세요" 같은 표현으로 끝내지 마세요.

검색 쿼리: {rewritten}

출력 형식 (번호와 텍스트만, 다른 텍스트 없이):
1. [서브쿼리 1]
2. [서브쿼리 2]
3. [서브쿼리 3]
""")


def decompose_query(rewritten: str, llm) -> list:
    """
    구체화된 쿼리 → 서브쿼리 리스트

    Args:
        rewritten : rewrite_query()의 결과
        llm       : LLM 인스턴스

    Returns:
        List[str]: 서브쿼리 리스트
    """
    chain = decompose_prompt | llm
    response = chain.invoke({"rewritten": rewritten}).content
    sub_queries = [
        q.strip().lstrip("1234567890. ")
        for q in response.split("\n")
        if q.strip() and q.strip()[0].isdigit()
    ]
    return sub_queries

## 5. Chained Pipeline
세 단계를 순서대로 이어서 실행하는 메인 **함수**

In [83]:
def get_chained_queries(summary: str, llm,
                        use_step_back: bool = True,
                        use_rewrite: bool = True,
                        use_decompose: bool = True) -> dict:
    """
    세 가지 변환을 순서대로 이어서 실행

    흐름: summary → step_back → rewritten → sub_queries

    Args:
        summary       : 원본 사용자 프로파일
        llm           : LLM 인스턴스
        use_step_back : Step-back Prompting 사용 여부 (기본 True)
        use_rewrite   : Query Rewriting 사용 여부 (기본 True)
        use_decompose : Sub-query Decomposition 사용 여부 (기본 True)

    Returns:
        {
            'step_back'  : str,       # Step-back 결과
            'rewritten'  : str,       # Rewriting 결과
            'sub_queries': List[str], # Decomposition 결과
            'all'        : List[str], # RRF에 넘길 전체 쿼리 목록
        }
    """
    # 1단계: Step-back → 의도 추상화
    step_back = step_back_query(summary, llm) if use_step_back else summary
    print(f"  [Step-back]  : {step_back}")

    # 2단계: Rewriting → 추상화된 의도를 구체적 검색어로
    rewritten = rewrite_query(step_back, llm) if use_rewrite else step_back
    print(f"  [Rewritten]  : {rewritten}")

    # 3단계: Decomposition → 구체적 쿼리를 조건 단위로 분해
    sub_queries = decompose_query(rewritten, llm) if use_decompose else []
    print(f"  [Sub-queries]: {sub_queries}")

    # 전체 쿼리 목록 (RRF 입력용)
    all_queries = [step_back, rewritten] + sub_queries

    return {
        "step_back":   step_back,
        "rewritten":   rewritten,
        "sub_queries": sub_queries,
        "all":         all_queries,
    }

## 6. 테스트 실행

In [84]:
# TEST_SUMMARY = "사용자는 한국 현대소설을 선호하며, 감성을 풍부하게 하고 다양한 시각을 배우고자 하는 목표를 가지고 있습니다. 현재 감성적이고 몽환적인 기분 속에서 잔잔한 음악과 함께 조용한 공간에서 독서를 즐기고 있으며, 스토리와 감정에 집중하는 스타일을 선호합니다. 중급 난이도의 작품을 통해 감정 정리와 아름다움을 느끼고자 하는 상황입니다."
TEST_SUMMARY = "사용자는 주말에 가볍게 읽으면서 스트레스를 해소하고 새로운 아이디어를 얻고자 하며, SF와 테크 스릴러 장르의 흥미로운 이야기를 선호합니다. 독서 스타일은 빠르게 전체적인 흐름을 파악하고, 스토리의 흡입력에 집중하는 중급자용입니다. 깊이 있는 분석보다는 이야기 전개와 흥미로운 설정에 중점을 두고 있으며, 가벼운 책을 통해 재미를 추구하고 있습니다."

print("[Query Transformations 실행]")
queries = get_chained_queries(TEST_SUMMARY, llm)

print(f"\n[전체 쿼리 목록] 총 {len(queries['all'])}개")
for i, q in enumerate(queries['all'], 1):
    print(f"  {i}. {q}")

[Query Transformations 실행]
  [Step-back]  : 사용자는 주말에 가벼운 독서를 통해 스트레스를 해소하고 창의적인 아이디어를 얻고자 하며, 흥미로운 스토리와 빠른 전개를 중시하는 SF 및 테크 스릴러 장르의 책을 선호합니다. 이들은 깊이 있는 분석보다는 몰입감 있는 이야기와 독특한 설정을 통해 재미를 추구하는 중급 독자입니다.
  [Rewritten]  : 중급 독자를 위한 가벼운 주말 독서로 적합한 흥미로운 스토리와 빠른 전개를 가진 몰입감 있는 SF 및 테크 스릴러 장르의 책, 독창적인 설정이 포함된 작품을 추천해 주세요.
  [Sub-queries]: ['중급 독자를 위한 몰입감 있는 SF 및 테크 스릴러 장르의 책을 찾고 있다.', '가벼운 주말 독서에 적합한 흥미로운 스토리와 빠른 전개를 가진 작품을 원한다.', '독창적인 설정이 포함된 책이 있을 경우, 해당 작품의 주제나 아이디어에 대해서도 알려주면 좋겠다.']

[전체 쿼리 목록] 총 5개
  1. 사용자는 주말에 가벼운 독서를 통해 스트레스를 해소하고 창의적인 아이디어를 얻고자 하며, 흥미로운 스토리와 빠른 전개를 중시하는 SF 및 테크 스릴러 장르의 책을 선호합니다. 이들은 깊이 있는 분석보다는 몰입감 있는 이야기와 독특한 설정을 통해 재미를 추구하는 중급 독자입니다.
  2. 중급 독자를 위한 가벼운 주말 독서로 적합한 흥미로운 스토리와 빠른 전개를 가진 몰입감 있는 SF 및 테크 스릴러 장르의 책, 독창적인 설정이 포함된 작품을 추천해 주세요.
  3. 중급 독자를 위한 몰입감 있는 SF 및 테크 스릴러 장르의 책을 찾고 있다.
  4. 가벼운 주말 독서에 적합한 흥미로운 스토리와 빠른 전개를 가진 작품을 원한다.
  5. 독창적인 설정이 포함된 책이 있을 경우, 해당 작품의 주제나 아이디어에 대해서도 알려주면 좋겠다.


## 7. Ablation Study용 — 단계별 조합 테스트

In [85]:
# # Sub-query만 사용
# print("[Sub-query only]")
# q1 = get_chained_queries(TEST_SUMMARY, llm,
#                           use_step_back=False,
#                           use_rewrite=False,
#                           use_decompose=True)

# # Step-back + Sub-query
# print("\n[Step-back + Sub-query]")
# q2 = get_chained_queries(TEST_SUMMARY, llm,
#                           use_step_back=True,
#                           use_rewrite=False,
#                           use_decompose=True)

# # 전체 (Step-back + Rewriting + Sub-query)
# print("\n[전체 파이프라인]")
# q3 = get_chained_queries(TEST_SUMMARY, llm,
#                           use_step_back=True,
#                           use_rewrite=True,
#                           use_decompose=True)

## 8. RRF 함수 정의

In [86]:
def reciprocal_rank_fusion(results_list: list, k: int = 60) -> list:
    """여러 검색 결과를 RRF로 결합. Returns: payload 딕셔너리 리스트 (점수 내림차순)"""
    scores, payloads = {}, {}
    for results in results_list:
        for rank, r in enumerate(results):
            isbn = r.payload.get("isbn", "")
            if isbn:
                scores[isbn]   = scores.get(isbn, 0) + 1 / (k + rank + 1)
                payloads[isbn] = r.payload
    return [payloads[isbn] for isbn, _ in sorted(scores.items(), key=lambda x: x[1], reverse=True)]

## 9. Query Transform RAG 노드

In [87]:
def query_transform_rag_node(state: CRSState) -> dict:
    summary    = state.get("summary", "")
    categories = state.get("genre_filter", [])

    # ── 1. Query Transformations ────────────────────────────
    print("\n[Query Transformations]")
    queries     = get_chained_queries(summary, llm)
    all_queries = queries["all"]

    # ── 2. 장르 필터 구성 (simple_rag_with_filtering과 동일) ──
    query_filter = None
    if categories:
        query_filter = Filter(
            must=[FieldCondition(key="cate_depth1", match=MatchAny(any=categories))]
        )

    # ── 3. 쿼리별 검색 ──────────────────────────────────────
    all_results = []
    for query in all_queries:
        query_vector = embedder.embed(query)
        if query_filter:
            results = db.search_with_filter(
                "books_v1", query_vector,
                query_filter=query_filter, limit=5, threshold=0.5,
            )
        else:
            results = db.search("books_v1", query_vector, limit=5, threshold=0.5)
        all_results.append(results)

        # print(query)
        # for r in results:
        #     print(f"score: {r.score}")
        #     print(f"title: {r.payload.get('title')}")
        #     print(f"book_intro: {r.payload.get('book_intro')}")
        # print("---------------")
    # ── 4. RRF 결합 ─────────────────────────────────────────
    merged_payloads = reciprocal_rank_fusion(all_results)

    # ── 5. 결과 정리 ─────────────────────────────────────────
    retrieved_books = [
        {
            "isbn":        p.get("isbn"),
            "title":       p.get("title"),
            "author":      p.get("author"),
            "book_intro":  p.get("book_intro"),
            "cate_depth1": p.get("cate_depth1"),
        }
        for p in merged_payloads[:10]
    ]

    # print(f"\n최종 검색 결과: {len(retrieved_books)}권")
    return {"retrieved_books": retrieved_books}

## 10. LLM 노드

*   항목 추가
*   항목 추가



In [88]:
def rag_llm_node(state: CRSState) -> dict:
    summary = state.get("summary", "")
    books   = state["retrieved_books"]

    context = "\n\n".join([
        f"ISBN: {b['isbn']}\n"
        f"제목: {b['title']}\n"
        f"저자: {b['author']}\n"
        f"장르: {b['cate_depth1']}\n"
        f"소개: {b['book_intro'][:100]}..."
        for b in books
    ])

    rag_prompt = ChatPromptTemplate.from_template("""
당신은 도서관 큐레이터 AI입니다.

[규칙]
- 반드시 [검색된 도서 목록]에 있는 책만 추천하세요.
- 반드시 JSON 형식으로만 답하세요. 다른 텍스트는 절대 포함하지 마세요.
- 사용자 프로파일을 참고해서 가장 적합한 도서 3권을 추천하세요.
- 추천 이유는 반드시 사용자 프로파일의 독서 목적, 성향, 상황과 연결해서 작성하세요.
- 장르는 반드시 [검색된 도서 목록]의 장르 값을 그대로 사용하세요.

[사용자 프로파일]
{summary}

[검색된 도서 목록]
{context}

[출력 형식]
[
    {{"title": "책 제목", "author": "저자", "isbn": "ISBN번호", "book_intro": "책 소개", "cate_depth1": "장르", "reason": "추천 이유 2~3문장"}},
    {{"title": "책 제목", "author": "저자", "isbn": "ISBN번호", "book_intro": "책 소개", "cate_depth1": "장르", "reason": "추천 이유 2~3문장"}},
    {{"title": "책 제목", "author": "저자", "isbn": "ISBN번호", "book_intro": "책 소개", "cate_depth1": "장르", "reason": "추천 이유 2~3문장"}}
]
""")

    chain    = rag_prompt | llm
    response = chain.invoke({"context": context, "summary": summary})

    try:
        recommendations = json.loads(response.content)
    except json.JSONDecodeError:
        recommendations = response.content

    return {
        "messages":       [AIMessage(content=response.content)],
        "recommendations": recommendations
    }

## 11. 그래프 구성 및 실행

In [92]:
graph_test = StateGraph(CRSState)
graph_test.add_node("genre", extract_genre_node)
graph_test.add_node("rag",   query_transform_rag_node)  # ← 변경된 노드
graph_test.add_node("llm",   rag_llm_node)
graph_test.add_edge(START,   "genre")
graph_test.add_edge("genre", "rag")
graph_test.add_edge("rag",   "llm")
graph_test.add_edge("llm",   END)
app_test = graph_test.compile()

# TEST_SUMMARY = "사용자는 한국 현대소설을 선호하며, 감성을 풍부하게 하고 다양한 시각을 배우고자 하는 목표를 가지고 있습니다. 현재 감성적이고 몽환적인 기분 속에서 잔잔한 음악과 함께 조용한 공간에서 독서를 즐기고 있으며, 스토리와 감정에 집중하는 스타일을 선호합니다. 중급 난이도의 작품을 통해 감정 정리와 아름다움을 느끼고자 하는 상황입니다."
TEST_SUMMARY = "사용자는 주말에 가볍게 읽으면서 스트레스를 해소하고 새로운 아이디어를 얻고자 하며, SF와 테크 스릴러 장르의 흥미로운 이야기를 선호합니다. 독서 스타일은 빠르게 전체적인 흐름을 파악하고, 스토리의 흡입력에 집중하는 중급자용입니다. 깊이 있는 분석보다는 이야기 전개와 흥미로운 설정에 중점을 두고 있으며, 가벼운 책을 통해 재미를 추구하고 있습니다."

result_test = app_test.invoke({
    "messages":        [HumanMessage(content=TEST_SUMMARY)],
    "summary":         TEST_SUMMARY,
    "retrieved_books": [],
    "recommendations": [],
    "genre_filter":    [],
})

print(result_test["messages"][-1].content)

추출된 장르: ['소설']

[Query Transformations]
  [Step-back]  : 사용자는 스트레스 해소와 새로운 아이디어 획득을 위해 흥미로운 스토리와 설정이 가득한 가벼운 SF 및 테크 스릴러 장르의 책을 선호합니다. 이 독자는 빠른 흐름 파악과 이야기의 몰입감에 집중하여 재미를 우선시하는 중급자 수준의 독서 스타일을 가지고 있습니다.
  [Rewritten]  : 중급자 수준의 독자를 위한 빠른 이야기 전개와 높은 몰입감을 제공하는, 스트레스 해소와 창의적인 아이디어 획득에 도움이 되는 가벼운 SF 및 테크 스릴러 장르의 흥미로운 스토리와 설정이 가득한 도서.
  [Sub-queries]: ['중급자 수준의 독자를 위한 빠른 이야기 전개와 높은 몰입감을 제공하는 가벼운 SF 및 테크 스릴러 장르의 도서를 찾고 싶다.', '스트레스 해소에 도움이 되는 흥미로운 스토리와 설정이 가득한 도서를 탐색하고 있다.', '창의적인 아이디어 획득에 기여할 수 있는 독특한 주제를 다룬 도서에 대한 추천을 원한다.']

최종 검색 결과: 10권
[
    {"title": "숨겨진 얼굴", "author": "이현종", "isbn": "9791190408752", "cate1": "소설", "reason": "이 스릴러는 반전과 흥미로운 소재로 가득 차 있어, 주말 동안 빠져들기 좋은 이야기입니다. 가족을 잃은 주인공의 이야기를 통해 감정의 깊이도 느낄 수 있지만, 전체적인 흐름이 매력적이어서 쉽게 읽을 수 있습니다."},
    {"title": "타임리미트 2", "author": "방지연", "isbn": "9788988820070", "cate1": "소설", "reason": "고정화된 틀을 뒤엎는 반전과 개성 있는 등장인물들이 매력적인 이 작품은 흡입력 있는 스토리로 독자에게 재미를 선사합니다. 중간중간 긴장감 넘치는 상황들이 연이어 펼쳐져서 지루할 틈이 없습니다."},
    {"title": "체포되셨습니다 2 - 홍염의 꽃", "auth

In [90]:
result_test["retrieved_books"]

[{'isbn': '9788959243532',
  'title': '체포되셨습니다 2 - 홍염의 꽃',
  'author': ['사이멜'],
  'book_intro': '제목에서 알 수 있듯이 본격 수사 판타지 소설. 그러나 프롤로그만 봐도 금세 알 수 있듯 복잡한 추리로 전개되는 머리 아픈 진행 따위는 없다. 다만 SKT 와 마찬가지로 개성 강한 여섯 명의 캐릭터들이 차분하게 사건을 해결해나가는 과정을 재미있게 그리고 있을 뿐이다. 거기에 작가 특유의 입담과 개그가 결합되어 자칫 무겁게만 진행될 수 있는 이 소설을 더욱 흥미진진하게 만든다.',
  'cate_depth1': ['소설']},
 {'isbn': '9791165459154',
  'title': '여행 그리고 파리 - Travel and Paris',
  'author': ['김나우'],
  'book_intro': '지친 일상에서 삶의 여유를 줄 수 있는 소설이다. 여행을 중심으로 주인공들의 사랑과 우정 가족애 신앙을 가지고 글을 썼고 독자들이 즐거운 시간이 되었으면 한다. 작가가 직접 여행에서 경험을 바탕으로 현실적인 배경과 정보도 있다.',
  'cate_depth1': ['소설']},
 {'isbn': '9788990019899',
  'title': '블루스카이 3 - 피로 얼룩진 혁명군',
  'author': ['박성재'],
  'book_intro': '탄탄한 구성이 돋보이는 소프트 어드벤처 판타지의 모델. 이제 판타지가 인류의 일그러진 미래로 눈을 돌렸다. 이 소설은 실패한 인류의 미래를 구원할 실낱같은 희망을 찾아나가고 있다. 간결하고 스피디한 문체로 결코 가볍지 않는 주제까지 아우르고 녹여낸 솜씨가 돋보이는 이 소설로 작가는 판타지가 나아갈 방향을 제시했다. 또한 판타지 문학의 세계를 확대 재생산하는 데 새로운 기틀을 마련했다.',
  'cate_depth1': ['소설']},
 {'isbn': '9788988820070',
  'title': '타임리미트 2',
  'au

## DeepEval 평가

In [91]:
# import logging
# logging.getLogger("deepeval").setLevel(logging.WARNING)

# from deepeval import evaluate
# from deepeval.metrics import AnswerRelevancyMetric, FaithfulnessMetric
# from deepeval.test_case import LLMTestCase

# test_cases = [
#     LLMTestCase(
#         input=TEST_SUMMARY,
#         actual_output="\n".join([
#             f"[{b['title']}] {b['reason']}"
#             for b in result["recommendations"]
#         ]),
#         retrieval_context=[
#             f"[{b['title']}] {b['book_intro']}"
#             for b in result["retrieved_books"]
#         ]
#     )
# ]

# evaluate(test_cases, [
#     AnswerRelevancyMetric(threshold=0.7, model="gpt-4o-mini", include_reason=True),
#     FaithfulnessMetric(threshold=0.7, model="gpt-4o-mini", include_reason=True),
# ])